<a href="https://colab.research.google.com/github/Romain357/tp_ingenierie-donnees/blob/main/python/backfill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Backfill AirPL

Version notebook du backfill massif pour tester et relancer le chargement historique.

In [4]:
import pandas as pd
import requests
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import bigquery
from tqdm.auto import tqdm

URL_MESURES_HORAIRES = "https://data.airpl.org/api/v1/mesure/horaire/"

def charger_dataframe_vers_bigquery(df, table_id, project_id="votre-projet-id", dataset_id="votre_dataset"):
    client = bigquery.Client(project=project_id)
    full_table_id = f"{project_id}.{dataset_id}.{table_id}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()
    print(f"\n✅ {len(df)} lignes chargées dans {table_id}")

def fetch_offset(offset, limit=1000):
    """Télécharge une page spécifique via son offset."""
    params = {"format": "json", "limit": limit, "offset": offset}
    try:
        response = requests.get(URL_MESURES_HORAIRES, params=params, timeout=30)
        if response.status_code == 200:
            return response.json().get("results", [])
    except Exception as e:
        pass
    return []

def lancer_backfill_ultra_rapide(date_limite_str, nb_pages_a_tenter=150, workers=25):
    date_limite = datetime.strptime(date_limite_str, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    all_results = []
    limit_per_page = 1000

    offsets = [i * limit_per_page for i in range(nb_pages_a_tenter)]

    print(f"🚀 Téléchargement de ~{nb_pages_a_tenter * limit_per_page} lignes avec {workers} threads...")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        future_to_offset = {executor.submit(fetch_offset, off, limit_per_page): off for off in offsets}

        # Utilisation de tqdm pour la barre de progression
        for future in tqdm(as_completed(future_to_offset), total=len(offsets), desc="Progression du téléchargement"):
            results = future.result()
            if not results:
                continue

            for ligne in results:
                try:
                    d = datetime.strptime(ligne["date_heure_tu"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
                    if d >= date_limite:
                        all_results.append(ligne)
                except:
                    continue

    if not all_results:
        print("❌ Aucune donnée trouvée.")
        return

    df = pd.DataFrame(all_results).drop_duplicates(subset=['id'])
    df = df.rename(columns={"code_commune": "insee_com", "date_heure_tu": "date_mesure"})

    cols = ["id", "date_mesure", "valeur", "code_station", "code_polluant", "insee_com"]
    df = df[cols].dropna(subset=["code_station", "code_polluant", "insee_com"])

    print(f"\n📊 Total récupéré : {len(df)} lignes uniques.")
    # charger_dataframe_vers_bigquery(df, "fait_mesures_heure", project_id="NOM_PROJET", dataset_id="NOM_DATASET")

if __name__ == "__main__":
    lancer_backfill_ultra_rapide("2025-01-01T00:00:00Z", nb_pages_a_tenter=150, workers=25)

🚀 Téléchargement de ~150000 lignes avec 25 threads...


Progression du téléchargement:   0%|          | 0/150 [00:00<?, ?it/s]


📊 Total récupéré : 149983 lignes uniques.
